# Notebook 1: Data Cleaning and Understanding
**Progetto:** Premier League Event Data Analysis

## 1. La sfida: Capire un dataset non documentato
Questo progetto parte da una sfida comune nel mondo reale del Data Science: abbiamo ricevuto un dataset massivo (~600k righe) in formato Parquet/CSV senza alcuna documentazione ufficiale. 

In questo notebook documentiamo il nostro processo di **"Archeologia dei Dati"**: abbiamo interrogato i valori concreti per ricostruire il significato delle colonne e la logica con cui il provider (Opta/WhoScored) ha registrato gli eventi della stagione Premier League 2020-21.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FILE_PATH = Path('../data/dataset.parquet')
df = pd.read_parquet(FILE_PATH)
print(f"Dataset caricato: {df.shape[0]:,} righe × {df.shape[1]} colonne")

## 2. Analisi delle Colonne — Cosa sono e come lo abbiamo capito

### Interpretazione delle variabili chiave:
*   **`outcomeType_displayName`**: Indica l'esito dell'azione (Successful / Unsuccessful).
*   **`isTouch`**: Fondamentale per distinguere le azioni reali. Se `False`, si tratta di eventi "amministrativi" (sostituzioni, fischi dell'arbitro).
*   **`isShot / isGoal / isOwnGoal`**: Presentano molti valori nulli. Abbiamo capito che si tratta di **Null Strutturali**: sono vuoti semplicemente perché l'evento non è un tiro o un gol. Non vanno imputati.
*   **`goalMouthY / goalMouthZ`**: Coordinate della porta (0-100%). Indicano dove il pallone ha attraversato la linea (es. Z=100 è la traversa, Z=0 è terra).
*   **`qualifiers`**: Campo complesso in formato JSON che contiene metadati come il piede usato o la zona del campo.

### La logica delle coordinate (x, y)
Lo spazio è misurato in **percentuali (0-100)** invece che in metri. Questo garantisce che i dati siano confrontabili tra stadi di dimensioni diverse. 
*   `x, y`: Punto di inizio dell'azione.
*   `endX, endY`: Punto di arrivo (solo per passaggi o tiri). Se un evento è puntuale (es. un fallo), queste colonne sono giustamente Null (Null strutturale).

## 3. Il Mistero dei due contatori (`eventId`)
Inizialmente abbiamo provato a usare `eventId` per identificare le partite, ma abbiamo scoperto un'anomalia: in alcune partite i contatori saltano improvvisamente da numeri piccoli a valori oltre il milione. 

**La nostra scoperta:** Il dataset sembra unire due flussi di dati diversi. Alcune squadre usano un contatore standard, altre partono da 1.000.000 per evitare collisioni. 

Per questo motivo, abbiamo identificato in **`Unnamed: 0`** la vera chiave: questo indice si resetta a 0 esattamente ad ogni fischio d'inizio di una nuova partita.

In [ ]:
# Ricostruzione del match_id basata sul reset dell'indice intra-partita
df = df.sort_index()
df['match_id'] = (df['Unnamed: 0'] == 0).cumsum()

print(f"Partite identificate: {df['match_id'].nunique()} (Target Premier League: 380)")

## 4. Analisi Temporale: Il problema del recupero
Abbiamo analizzato due colonne temporali:
1.  **`minute`**: Il cronometro classico dell'arbitro (si ferma al 45° e al 90°).
2.  **`expandedMinute`**: Il minuto reale di gioco che include il recupero (es. un gol al 93°).

**Valore Sentinella 32767:** Abbiamo notato che alcuni eventi hanno minuto 32767. Si tratta del valore massimo per un intero a 16 bit, usato dal sistema come placeholder per eventi fuori dal tempo di gioco (PreMatch o PostGame).

In [ ]:
print("--- Distribuzione dei Periodi ---")
print(df['period_displayName'].value_counts())

# Filtro per tenere solo il calcio giocato (First e Second Half)
df_clean = df[df['period_displayName'].isin(['FirstHalf', 'SecondHalf'])].copy()

print(f"\nRighe rimosse (eventi di sistema): {len(df) - len(df_clean):,}")

## 5. Conclusione e Salvataggio (Silver Layer)
Dopo aver compreso la struttura e rimosso le anomalie, salviamo il dataset pulito. Questo file sarà la base per l'estrazione delle feature nel Notebook 2.

In [ ]:
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(OUT_DIR / 'dataset_clean.parquet', engine='pyarrow', compression='snappy')
print("Dataset Silver (pulito) salvato con successo!")